# StyleGAN2 — Visualisation

**Ce notebook ne fait pas d'entraînement.**

Il charge le checkpoint et permet de :
1. Voir les images générées
2. Analyser les courbes de loss
3. Interpoler entre deux visages
4. Style mixing
5. Explorer l'espace W (préparation analyse attributs)

## 1 — Setup & Chargement du checkpoint

In [ ]:
import sys, os, torch
import matplotlib.pyplot as plt

STYLEGAN2_DIR = os.path.abspath('.')
ROOT          = os.path.abspath('..')
for p in [STYLEGAN2_DIR, ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

import config as cfg
import importlib; importlib.reload(cfg)
from model import build_stylegan2
from utils.visualizer import show_grid

# Charger le checkpoint
ckpt_path = os.path.join(cfg.CHECKPOINTS_DIR, 'stylegan2_final.pt')
ckpt      = torch.load(ckpt_path, map_location=device)

# G_ema = version stable pour toutes les visualisations
G, D = build_stylegan2(cfg.Z_DIM, cfg.W_DIM, cfg.CHANNELS, device)
G.load_state_dict(ckpt['G_state_dict'])
G_ema, _ = build_stylegan2(cfg.Z_DIM, cfg.W_DIM, cfg.CHANNELS, device)
G_ema.load_state_dict(ckpt['G_ema_state_dict'])
G_ema.eval()

# Truncation mean : moyenne de 10 000 vecteurs w
with torch.no_grad():
    z_many = torch.randn(10000, cfg.Z_DIM).to(device)
    W_MEAN = G_ema.mapping(z_many).mean(0, keepdim=True)

PSI = 0.7  # Truncation factor : 0 = visages moyens nets, 1 = diversité max

print(f'Device  : {device}')
print(f'Epoch   : {ckpt["epoch"]}')
print(f'Mode    : {ckpt["config"]["mode"]}')
print(f'W shape : {ckpt["w_samples"].shape}')
print(f'PSI     : {PSI}')
print('Checkpoint chargé — G_ema prêt.')

## 2 — Courbes de loss

Un bon entraînement : G et D oscillent autour de valeurs stables.
Si D → 0 : mode collapse. Si G explose : instabilité.

In [ ]:
losses_G = ckpt['losses_G']
losses_D = ckpt['losses_D']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses_G, label='Générateur',     color='royalblue')
ax.plot(losses_D, label='Discriminateur', color='tomato')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Évolution des pertes — StyleGAN2')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Loss finale G : {losses_G[-1]:.4f}')
print(f'Loss finale D : {losses_D[-1]:.4f}')

## 3 — Génération d'images

Utilise **G_ema** (version stable) avec le **truncation trick** (PSI=0.7).

- PSI proche de 0 → visages très nets mais peu variés (proches de la moyenne)
- PSI = 1 → diversité maximale mais quelques artefacts possibles
- PSI = 0.7 → meilleur compromis qualité/diversité

Change le `SEED` pour explorer différents visages.

In [ ]:
SEED = 42  # change ce nombre pour voir d'autres visages

torch.manual_seed(SEED)
with torch.no_grad():
    z    = torch.randn(16, cfg.Z_DIM).to(device)
    w    = G_ema.mapping(z)
    w    = W_MEAN + PSI * (w - W_MEAN)   # truncation trick
    imgs = G_ema.synthesis(w)

show_grid(imgs, title=f'StyleGAN2 — 16 visages (seed={SEED}, PSI={PSI})')

## 4 — Interpolation

Transition progressive entre deux visages.
Alpha = 0 → visage A pur. Alpha = 1 → visage B pur.

In [ ]:
SEED_A = 42
SEED_B = 99
STEPS  = 8   # nombre d'étapes dans la transition

torch.manual_seed(SEED_A)
z_a = torch.randn(1, cfg.Z_DIM).to(device)
torch.manual_seed(SEED_B)
z_b = torch.randn(1, cfg.Z_DIM).to(device)

interpolated = []
with torch.no_grad():
    w_a = G_ema.mapping(z_a)
    w_b = G_ema.mapping(z_b)
    # Interpolation dans l'espace W (plus cohérente que dans Z)
    for alpha in torch.linspace(0, 1, STEPS):
        w_interp = (1 - alpha) * w_a + alpha * w_b
        w_interp = W_MEAN + PSI * (w_interp - W_MEAN)  # truncation
        img      = G_ema.synthesis(w_interp)
        interpolated.append(img)

imgs = torch.cat(interpolated, dim=0)
show_grid(imgs, title=f'Interpolation W-space : seed {SEED_A} → seed {SEED_B}', n=STEPS)

## 5 — Style Mixing

Mélange les caractéristiques de deux visages.
Fonctionnalité exclusive à StyleGAN2 grâce à l'espace W.

In [ ]:
SEED_A = 42
SEED_B = 99

torch.manual_seed(SEED_A)
z_a = torch.randn(1, cfg.Z_DIM).to(device)
torch.manual_seed(SEED_B)
z_b = torch.randn(1, cfg.Z_DIM).to(device)

with torch.no_grad():
    w_a   = G_ema.mapping(z_a)
    w_b   = G_ema.mapping(z_b)
    w_mix = 0.5 * w_a + 0.5 * w_b

    # Truncation sur chaque vecteur
    w_a_t   = W_MEAN + PSI * (w_a   - W_MEAN)
    w_b_t   = W_MEAN + PSI * (w_b   - W_MEAN)
    w_mix_t = W_MEAN + PSI * (w_mix - W_MEAN)

    img_a   = G_ema.synthesis(w_a_t)
    img_b   = G_ema.synthesis(w_b_t)
    img_mix = G_ema.synthesis(w_mix_t)

def to_img(t):
    return (t.squeeze().cpu() * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Style Mixing — StyleGAN2 (G_ema + truncation)', fontsize=13)
axes[0].imshow(to_img(img_a));   axes[0].set_title(f'Visage A (seed {SEED_A})'); axes[0].axis('off')
axes[1].imshow(to_img(img_mix)); axes[1].set_title('Mix A + B (W-space)');        axes[1].axis('off')
axes[2].imshow(to_img(img_b));   axes[2].set_title(f'Visage B (seed {SEED_B})'); axes[2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(cfg.GENERATED_IMGS_DIR, 'style_mixing.png'), dpi=100)
plt.show()

## 6 — Exploration de l'espace W

Aperçu des vecteurs W collectés pendant l'entraînement.
Ces vecteurs seront utilisés plus tard pour trouver les directions
d'attributs (âge, sourire, genre...) via PCA.

In [ ]:
w_samples = ckpt['w_samples']  # shape (N, 512)
print(f'Vecteurs W disponibles : {w_samples.shape}')
print(f'Moyenne : {w_samples.mean():.4f}')
print(f'Std     : {w_samples.std():.4f}')

# Distribution des W — doit être plus concentrée que z (espace gaussien)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(w_samples[:, :10].numpy().flatten(), bins=50, color='royalblue', alpha=0.7)
axes[0].set_title('Distribution espace W (10 premières dims)')
axes[0].set_xlabel('Valeur')

# Norme des vecteurs W
norms = w_samples.norm(dim=1).numpy()
axes[1].hist(norms, bins=50, color='tomato', alpha=0.7)
axes[1].set_title('Norme des vecteurs W')
axes[1].set_xlabel('||w||')

plt.tight_layout()
plt.show()
print('Prêt pour analyse PCA des attributs après entraînement RunPod.')